# DrugBank exploration

This notebook presents the four study figures and the code that reproduces them. The committed figures are displayed below without requiring DrugBank data.

Once you have the appropriate licences and have prepared the data using the [data guide](../../data/README.md) and [DataPreparation.ipynb](../../data/DataPreparation.ipynb), you can regenerate the figures below using the plotting functions in [src/dtpkg/drugbank](../../src/dtpkg/drugbank/).

Two populations are used:

- **Modality coverage:** all small-molecule and biotech entries in the DrugBank snapshot, regardless of approval status.
- **MeSH exploration:** approved drugs whose annotations map to branch D (Chemicals and Drugs) of the 2025 MeSH descriptor hierarchy. Ancestors are propagated within each tree-number path. The study snapshot contains 3,342 such drugs.

MeSH knowledge categories describe each drug's deepest annotation. Embedding scopes describe which terms are retained across the entire mapped-drug population. These are different quantities.


## Paths and run options

Replace `...` with your quoted checkout and prepared-data paths.
See the [setup instructions](../../README.md) for dependencies.
Keep `REGENERATE = False` to view the supplied figures, or set it to `True`
to redraw them from the tables prepared in [DataPreparation.ipynb](../../data/DataPreparation.ipynb).
Generated figures go to `OUTPUT_DIR`.


In [ ]:
from pathlib import Path
import sys

# Replace each ... with your own quoted path or Path(...).
PROJECT_ROOT = ...  # DTP-KG checkout folder
DATA_DIR = ...  # Prepared private data folder

# Check and normalize the paths supplied above.
if any(path is ... for path in (PROJECT_ROOT, DATA_DIR)):
    raise ValueError("Replace ... in PROJECT_ROOT, DATA_DIR with your local paths.")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()

# Import this checkout's source files.
SOURCE_DIR = PROJECT_ROOT / "src"
if not (SOURCE_DIR / "dtpkg" / "__init__.py").is_file():
    raise ValueError("PROJECT_ROOT must point to the DTP-KG checkout.")
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

DATA_DIR = Path(DATA_DIR).expanduser().resolve()

# Prepared inputs from data/DataPreparation.ipynb.
COVERAGE_CSV = DATA_DIR / "drugbank" / "coverage_analysis_cache.csv"
ANNOTATIONS_CSV = DATA_DIR / "mesh" / "extended_drug_info.csv"
OUTPUT_DIR = PROJECT_ROOT / "experiments" / "01_data_exploration" / "figures" / "reproduced"

REGENERATE = False


In [2]:
from dtpkg.drugbank.generate_modality_coverage_figure import generate_modality_coverage
from dtpkg.drugbank.generate_level_dist_figure import generate_level_distribution
from dtpkg.drugbank.generate_mesh_freq_cov_figures import generate_mesh_panels

coverage = annotations = None


## Load the prepared analysis tables

[DataPreparation.ipynb](../../data/DataPreparation.ipynb) creates the modality-coverage table
and ancestor-expanded MeSH annotations. This notebook reads those tables to
reproduce the figures below.


In [3]:
if REGENERATE:
    import pandas as pd
    from dtpkg.mesh_scopes import load_annotations

    missing = [str(path) for path in (COVERAGE_CSV, ANNOTATIONS_CSV) if not path.is_file()]
    if missing:
        raise FileNotFoundError(
            "Run data/DataPreparation.ipynb first; missing inputs:\n" + "\n".join(missing)
        )

    coverage = pd.read_csv(COVERAGE_CSV)
    annotations = load_annotations(ANNOTATIONS_CSV)
    print(f"Modality population: {len(coverage):,} drugs")
    print(f"Approved drugs mapped to MeSH branch D: {annotations['drugbank_id'].nunique():,}")


## Modality coverage

Each percentage is the fraction of entries of that drug type with a present annotation. The composite “Any KG-representable edge” is the union of targets, drug interactions, enzymes, transporters, carriers, pathways, and reactions. The top ten candidate modalities are ranked by biotech coverage, with small-molecule coverage breaking ties. Amino-acid sequences and SMILES are shown separately as structural baselines.

![Modality coverage in DrugBank](figures/modality_coverage_table.png)


In [4]:
if REGENERATE:
    path = generate_modality_coverage(coverage, OUTPUT_DIR)
    from IPython.display import Image, display
    display(Image(filename=str(path)))


## Deepest MeSH level per drug

The deepest ancestor-closed annotation determines each drug's knowledge category: Low (levels 1–5), Mid (6–7), or Deep (8–10). The figure checks that these assignments reproduce the stored annotation columns. Each drug is counted once.

![Distribution of deepest MeSH levels](figures/level_dist.png)


In [5]:
if REGENERATE:
    path = generate_level_distribution(annotations, OUTPUT_DIR)
    display(Image(filename=str(path)))


## MeSH term frequency across embedding scopes

A term is identified by its MeSH **tree number**. Its frequency is the number of distinct mapped drugs carrying that term after applying the scope's depth range, before frequency filtering. The density is computed on `log10(drugs per term)` with the same kernel bandwidth as the study figure.

| Embedding scope | Term depth | Retained frequency, inclusive |
|---|---|---|
| Shallow | 1–5 | 2–35 drugs |
| Intermediate | 1–7 | 2–63 drugs |
| Full | 1–10 | 2–52 drugs |

These fixed study parameters come from the shared `dtpkg.mesh_scopes` module. Comparing and selecting embedding scopes belongs to the next analysis task; this notebook describes their annotation coverage.

![MeSH term-frequency distributions](figures/mesh_term_freq_panel.png)


In [6]:
if REGENERATE:
    mesh_figures = generate_mesh_panels(annotations, OUTPUT_DIR)
    display(Image(filename=str(mesh_figures["mesh_term_freq_panel"])))


## Cumulative drug coverage

Terms are added in decreasing frequency order. Faded curves include all terms in the depth range; bold curves include only the retained vocabulary. Both use the same denominator: drugs with at least one term in the scope's depth range (3,342 in the study snapshot). Ancestor propagation makes this denominator the same for all three scopes.

The retained vocabularies contain 1,416 / 2,128 / 2,183 terms and cover 2,694 / 3,125 / 2,994 drugs, respectively. The plotting function checks these snapshot counts. Saved TF-IDF/SVD embeddings are **not required** to reproduce the curves; checking their drug sets is an optional additional validation.

![Cumulative MeSH drug coverage](figures/mesh_term_cov_panel.png)


In [7]:
if REGENERATE:
    display(Image(filename=str(mesh_figures["mesh_term_cov_panel"])))
